# Attention-MADE on a Colab GPU

Connect this notebook to a **Colab GPU** kernel (Select Kernel → Colab → pick GPU, not CPU), then run the cells in order.

This clones the `made-attention` branch, installs the extra packages Colab does not ship, and trains the residual LayerNorm attention-MADE candidate. Checkpoints land in `outputs/made/binarized_mnist_attention/` on the runtime (they disappear when the VM is recycled).

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. In Cursor: Select Kernel → Colab → choose a GPU runtime, then rerun."
)
print(torch.cuda.get_device_name(0))
print("torch", torch.__version__)

In [ ]:
from pathlib import Path

REPO = "https://github.com/ml-and-ds-degree/deep-generative-models-of-texts-and-images.git"
BRANCH = "made-attention"
ROOT = Path("/content/deep-generative-models-of-texts-and-images")

if not (ROOT / "src" / "made_reproduction").exists():
    !git clone --branch {BRANCH} --depth 1 "{REPO}" "{ROOT}"

%cd {ROOT}
!git rev-parse --abbrev-ref HEAD && git log -1 --oneline

In [ ]:
from pathlib import Path
import os

%pip install -q lightning cyclopts torchmetrics

os.environ["PYTHONPATH"] = str(Path.cwd() / "src")
print("PYTHONPATH", os.environ["PYTHONPATH"])

In [ ]:
!PYTHONPATH=src python -m made_reproduction.cli train binarized-mnist \
  --architecture attention \
  --accelerator gpu \
  --max-epochs 25 \
  --num-workers 2

In [ ]:
from pathlib import Path

ckpt_dir = Path("outputs/made/binarized_mnist_attention/checkpoints")
ckpts = sorted(ckpt_dir.glob("epoch-*.ckpt"))
assert ckpts, "No epoch checkpoint yet; wait for training to finish."
best = ckpts[-1]
print("evaluating", best)

!PYTHONPATH=src python -m made_reproduction.cli evaluate "{best}" binarized-mnist --accelerator gpu